# Stage 2 Answer Generation

Этот notebook готовит batch для ответа на вопросы по таблицам.

Вход:
- JSONL с вопросами
- для каждого вопроса массив id точек

Пайплайн:
1. Загружает вопросы из JSONL
2. По списку id получает тексты фрагментов таблиц из хранилища
3. Собирает промпт без упоминания внешнего retrieval-движка
4. Запускает локальную генерацию через vLLM
5. Сохраняет JSONL и CSV с answer, reasoning, supporting_rows, supporting_columns

В prompt используется только естественное описание вопроса и 10 таблиц-контекстов.

In [ ]:
%pip install qdrant_client vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.4/244.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 168.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 12

In [ ]:
from __future__ import annotations

import ast
import json
import math
import os
import random
import re
from collections import OrderedDict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
from qdrant_client import QdrantClient
from tqdm.auto import tqdm

INPUT_JSONL = Path(os.getenv('STAGE2_INPUT_JSONL', 'stage2_eval_data.jsonl'))
OUTPUT_JSONL = Path(os.getenv('STAGE2_OUTPUT_JSONL', 'stage2_model_answers.jsonl'))
OUTPUT_CSV = Path(os.getenv('STAGE2_OUTPUT_CSV', 'stage2_model_answers.csv'))

QDRANT_HOST = os.getenv('QDRANT_HOST')
QDRANT_PORT = int(os.getenv('QDRANT_PORT', '80'))
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY') or None
QDRANT_COLLECTION = os.getenv('QDRANT_COLLECTION')

VLLM_MODEL = os.getenv('VLLM_MODEL', 'Qwen/Qwen3-8B')
VLLM_DTYPE = os.getenv('VLLM_DTYPE', 'bfloat16')
VLLM_MAX_MODEL_LEN = int(os.getenv('VLLM_MAX_MODEL_LEN', '15360'))
VLLM_MAX_NUM_SEQS = int(os.getenv('VLLM_MAX_NUM_SEQS', '128'))
VLLM_TENSOR_PARALLEL_SIZE = int(os.getenv('VLLM_TENSOR_PARALLEL_SIZE', '1'))
MODEL_BATCH_SIZE = int(os.getenv('MODEL_BATCH_SIZE', '128'))
MAX_GENERATED_TOKENS = int(os.getenv('MAX_GENERATED_TOKENS', '512'))
QDRANT_RETRIEVE_BATCH_SIZE = int(os.getenv('QDRANT_RETRIEVE_BATCH_SIZE', '128'))
MAX_EVIDENCE_CHARS = int(os.getenv('MAX_EVIDENCE_CHARS', '1800'))
DATASET_LIMIT = int(os.getenv('DATASET_LIMIT', '10000')) or None
RANDOM_SEED = int(os.getenv('RANDOM_SEED', '42'))

TEXT_PAYLOAD_KEYS = [
    'text',
    'content',
    'chunk_text',
    'row_text',
    'table_text',
    'serialized_text',
    'page_content',
    'markdown',
    'body',
]

QUESTION_TYPE_FALLBACK = 'unknown'
print('Config loaded')

Config loaded


In [ ]:
@dataclass(frozen=True)
class Stage2Example:
    qid: str
    question: str
    rag_answers: list[str]
    question_meta: dict[str, Any]
    table_meta: dict[str, Any]
    raw: dict[str, Any]

def make_qdrant_client() -> QdrantClient:
    return QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT, api_key=QDRANT_API_KEY, https=False)

def chunked(values: list[Any], size: int) -> Iterable[list[Any]]:
    size = max(1, int(size))
    for start in range(0, len(values), size):
        yield values[start:start + size]

def normalize_text(value: Any) -> str:
    if value is None:
        return ''
    if isinstance(value, float) and pd.isna(value):
        return ''
    if isinstance(value, str):
        return re.sub(r'\s+', ' ', value).strip()
    return re.sub(r'\s+', ' ', str(value)).strip()

def normalize_row_ids(value: Any) -> list[int]:
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        out: list[int] = []
        for item in value:
            try:
                out.append(int(item))
            except Exception:
                continue
        return out
    try:
        return [int(value)]
    except Exception:
        return []

def load_stage2_examples(path: Path = INPUT_JSONL) -> list[Stage2Example]:
    examples: list[Stage2Example] = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            raw = json.loads(line)
            rag_answers = [str(x).strip() for x in raw.get('rag_answers', []) if str(x).strip()]
            if not rag_answers:
                continue
            question_meta = raw.get('question_meta') or {}
            table_meta = raw.get('table_meta') or {}
            examples.append(
                Stage2Example(
                    qid=str(raw.get('qid', '')).strip(),
                    question=normalize_text(raw.get('question')),
                    rag_answers=rag_answers,
                    question_meta=question_meta if isinstance(question_meta, dict) else {},
                    table_meta=table_meta if isinstance(table_meta, dict) else {},
                    raw=raw,
                )
            )
    return examples

def retrieve_points_by_id(
    client: QdrantClient,
    collection_name: str,
    point_ids: list[str],
) -> dict[str, dict[str, Any]]:
    point_ids = list(OrderedDict.fromkeys(str(pid).strip() for pid in point_ids if str(pid).strip()))
    payload_by_id: dict[str, dict[str, Any]] = {}

    for batch in tqdm(list(chunked(point_ids, QDRANT_RETRIEVE_BATCH_SIZE)), desc='Retrieving table fragments'):
        points = client.retrieve(
            collection_name=collection_name,
            ids=batch,
            with_payload=True,
            with_vectors=False,
        )
        for point in points:
            payload_by_id[str(point.id)] = point.payload or {}

    return payload_by_id

def payload_text(payload: dict[str, Any]) -> str:
    if not isinstance(payload, dict) or not payload:
        return ''

    for key in TEXT_PAYLOAD_KEYS:
        value = payload.get(key)
        if isinstance(value, str):
            text = normalize_text(value)
            if text:
                return text
        if isinstance(value, list):
            joined = ' '.join(normalize_text(item) for item in value if normalize_text(item))
            if joined:
                return joined

    string_values: list[str] = []
    for value in payload.values():
        if isinstance(value, str):
            text = normalize_text(value)
            if text:
                string_values.append(text)
    if string_values:
        return max(string_values, key=len)

    return json.dumps(payload, ensure_ascii=False)

def compact_point_payload(point_id: str, payload: dict[str, Any]) -> dict[str, Any]:
    text = payload_text(payload)
    if len(text) > MAX_EVIDENCE_CHARS:
        text = text[:MAX_EVIDENCE_CHARS].rstrip() + ' ...'

    return {
        'point_id': str(point_id),
        'chunk_id': payload.get('chunk_id'),
        'table_id': payload.get('table_id'),
        'row_ids': normalize_row_ids(payload.get('row_ids') or payload.get('supporting_rows')),
        'text': text,
    }

def build_evidence_block(example: Stage2Example, payload_by_id: dict[str, dict[str, Any]]) -> tuple[str, list[dict[str, Any]]]:
    evidence_points: list[dict[str, Any]] = []
    texts: list[str] = []

    for point_id in example.rag_answers:
        payload = payload_by_id.get(str(point_id), {})
        compact = compact_point_payload(point_id, payload)
        evidence_points.append(compact)

        text = compact['text'] or ''
        if text:
            texts.append(text)

    return '\n\n'.join(texts), evidence_points

print('Qdrant helpers ready')

Qdrant helpers ready


In [ ]:
SYSTEM_PROMPT = """Ты отвечаешь на вопрос по таблице, используя только предоставленные фрагменты контекста.

Верни строго один JSON-объект без markdown и без поясняющего текста вне JSON.

Формат ответа:
{
  "answer": "...",
  "reasoning": "...",
  "supporting_rows": [0, 1],
  "supporting_columns": ["col_a", "col_b"]
}

Правила:
- answer должен быть точным и опираться только на evidence.
- reasoning должен кратко объяснять ход вывода.
- supporting_rows бери из row_ids в evidence, если они действительно использованы.
- supporting_columns перечисляй только те столбцы, которые реально помогли ответить.
- если данных недостаточно, верни answer как null, supporting_rows как [], supporting_columns как [], а в reasoning напиши, чего не хватает."""


def build_prompt(example: Stage2Example, evidence_block: str) -> str:
    article_title = normalize_text((example.table_meta.get('page') or {}).get('title'))
    table_title = normalize_text(example.table_meta.get('title'))
    question_type = normalize_text(example.question_meta.get('question_type')) or QUESTION_TYPE_FALLBACK

    return f"""
{SYSTEM_PROMPT}

Вопрос: {example.question}
Тип вопроса: {question_type}
Статья: {article_title}
Таблица: {table_title}

Контекст таблиц:
{evidence_block}

Верни только JSON с полями answer, reasoning, supporting_rows, supporting_columns.
""".strip()


_JSON_OBJECT_RE = re.compile(r"\{.*\}", re.S)


def parse_model_json(text: str) -> dict[str, Any]:
    raw_text = normalize_text(text)
    if raw_text.startswith("```"):
        raw_text = raw_text.strip("`")
        raw_text = raw_text.replace("json", "", 1).strip()

    match = _JSON_OBJECT_RE.search(raw_text)
    candidate = match.group(0) if match else raw_text

    try:
        parsed = json.loads(candidate)
    except Exception:
        parsed = ast.literal_eval(candidate)

    if not isinstance(parsed, dict):
        raise ValueError('model output is not a JSON object')
    return parsed


def make_llm():
    from vllm import LLM  # pyright: ignore[reportMissingImports]

    return LLM(
        model=VLLM_MODEL,
        dtype=VLLM_DTYPE,
        trust_remote_code=True,
        tensor_parallel_size=VLLM_TENSOR_PARALLEL_SIZE,
        max_model_len=VLLM_MAX_MODEL_LEN,
        max_num_seqs=VLLM_MAX_NUM_SEQS,
        enable_prefix_caching=True,
    )


def make_sampling_params():
    from vllm import SamplingParams  # pyright: ignore[reportMissingImports]

    return SamplingParams(temperature=0.0, top_p=1.0, max_tokens=MAX_GENERATED_TOKENS)


def build_model_row(
    example: Stage2Example,
    prompt: str,
    evidence_points: list[dict[str, Any]],
    raw_response: str,
    parsed_response: dict[str, Any],
) -> dict[str, Any]:
    question_meta = example.question_meta or {}
    answer = parsed_response.get('answer')
    reasoning = parsed_response.get('reasoning')
    supporting_rows = parsed_response.get('supporting_rows')
    supporting_columns = parsed_response.get('supporting_columns')

    if not isinstance(supporting_rows, list):
        supporting_rows = []
    if not isinstance(supporting_columns, list):
        supporting_columns = []

    return {
        'qid': example.qid,
        'question': example.question,
        'question_type': normalize_text(question_meta.get('question_type')) or QUESTION_TYPE_FALLBACK,
        'article_title': normalize_text((example.table_meta.get('page') or {}).get('title')),
        'table_title': normalize_text(example.table_meta.get('title')),
        'point_ids': example.rag_answers,
        'evidence_points': evidence_points,
        'prompt': prompt,
        'model_raw': raw_response,
        'answer': answer,
        'reasoning': reasoning,
        'supporting_rows': supporting_rows,
        'supporting_columns': supporting_columns,
        'reference_answer': question_meta.get('answer'),
        'reference_reasoning': question_meta.get('reasoning'),
        'reference_supporting_rows': question_meta.get('supporting_rows'),
        'reference_supporting_columns': question_meta.get('supporting_columns'),
        'status': 'ok',
    }


print('Prompt and parsing helpers ready')

Prompt and parsing helpers ready


In [ ]:
def generate_batch(llm, batch_examples: list[Stage2Example], payload_by_id: dict[str, dict[str, Any]]) -> list[dict[str, Any]]:
    prompts: list[str] = []
    prompt_token_ids: list[list[int]] = []
    evidence_points_list: list[list[dict[str, Any]]] = []
    tokenizer = llm.get_tokenizer()
    max_prompt_tokens = max(1, VLLM_MAX_MODEL_LEN - MAX_GENERATED_TOKENS)

    for example in batch_examples:
        evidence_block, evidence_points = build_evidence_block(example, payload_by_id)
        prompt = build_prompt(example, evidence_block)
        prompts.append(prompt)
        prompt_token_ids.append(tokenizer.encode(prompt, truncation=True, max_length=max_prompt_tokens))
        evidence_points_list.append(evidence_points)

    sampling_params = make_sampling_params()
    outputs = llm.generate(prompt_token_ids, sampling_params=sampling_params)
    rows: list[dict[str, Any]] = []

    for example, prompt, evidence_points, output in zip(batch_examples, prompts, evidence_points_list, outputs):
        raw_response = ''
        if getattr(output, 'outputs', None):
            raw_response = output.outputs[0].text or ''

        try:
            parsed = parse_model_json(raw_response)
            rows.append(build_model_row(example, prompt, evidence_points, raw_response, parsed))
        except Exception as exc:
            question_meta = example.question_meta or {}
            rows.append({
                'qid': example.qid,
                'question': example.question,
                'question_type': normalize_text(question_meta.get('question_type')) or QUESTION_TYPE_FALLBACK,
                'article_title': normalize_text((example.table_meta.get('page') or {}).get('title')),
                'table_title': normalize_text(example.table_meta.get('title')),
                'point_ids': example.rag_answers,
                'evidence_points': evidence_points,
                'prompt': prompt,
                'model_raw': raw_response,
                'answer': None,
                'reasoning': f'parse_error: {exc}',
                'supporting_rows': [],
                'supporting_columns': [],
                'reference_answer': question_meta.get('answer'),
                'reference_reasoning': question_meta.get('reasoning'),
                'reference_supporting_rows': question_meta.get('supporting_rows'),
                'reference_supporting_columns': question_meta.get('supporting_columns'),
                'status': 'parse_error',
            })

    return rows

def reset_output_files() -> None:
    OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_JSONL.write_text('', encoding='utf-8')
    OUTPUT_CSV.write_text('', encoding='utf-8')

def append_outputs(rows: list[dict[str, Any]], write_csv_header: bool = False) -> None:
    OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

    with OUTPUT_JSONL.open('a', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False, mode='a', header=write_csv_header)

def sample_examples(examples: list[Stage2Example], limit: int | None, seed: int) -> list[Stage2Example]:
    if limit is None or limit >= len(examples):
        return examples

    rng = random.Random(seed)
    sampled_indices = sorted(rng.sample(range(len(examples)), limit))
    return [examples[index] for index in sampled_indices]

def run_pipeline(limit: int | None = DATASET_LIMIT, seed: int = RANDOM_SEED) -> list[dict[str, Any]]:
    examples = load_stage2_examples(INPUT_JSONL)
    examples = sample_examples(examples, limit, seed)

    if not examples:
        raise ValueError('No examples loaded from stage2 input JSONL')

    point_ids: list[str] = []
    for example in examples:
        point_ids.extend(example.rag_answers)

    client = make_qdrant_client()
    payload_by_id = retrieve_points_by_id(client, QDRANT_COLLECTION, point_ids)

    missing = [pid for pid in OrderedDict.fromkeys(point_ids) if pid not in payload_by_id]
    if missing:
        print(f'[WARN] Missing {len(missing)} table fragments, example ids: {missing[:10]}')

    llm = make_llm()
    reset_output_files()
    rows: list[dict[str, Any]] = []

    total_batches = math.ceil(len(examples) / MODEL_BATCH_SIZE)
    for batch_index, start_idx in enumerate(tqdm(range(0, len(examples), MODEL_BATCH_SIZE), total=total_batches, desc='Generating batches'), 1):
        batch_examples = examples[start_idx:start_idx + MODEL_BATCH_SIZE]
        print(f'Batch {batch_index}/{total_batches}: {len(batch_examples)} questions')
        batch_rows = generate_batch(llm, batch_examples, payload_by_id)
        rows.extend(batch_rows)
        append_outputs(batch_rows, write_csv_header=(batch_index == 1))

    return rows

print('Runner ready')

Runner ready


In [ ]:
examples = load_stage2_examples(INPUT_JSONL)
print(f'Loaded examples: {len(examples)}')

sample = examples[0]
client = make_qdrant_client()
payload_by_id = retrieve_points_by_id(client, QDRANT_COLLECTION, sample.rag_answers)
evidence_block, evidence_points = build_evidence_block(sample, payload_by_id)
prompt = build_prompt(sample, evidence_block)

print(prompt[:6000])
print('--- evidence points ---')
print(json.dumps(evidence_points[:2], ensure_ascii=False, indent=2))

In [ ]:
import os
#os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"

In [ ]:
results = run_pipeline()
print(f'Wrote {len(results)} rows to {OUTPUT_JSONL}')
pd.DataFrame(results).head()